# 02. 이커머스 퍼널 분석 (Ecommerce Funnel Analysis)

**Dataset**: Google Merchandise Store (GA4 Public Dataset)  
**Period**: 2016-08-01 ~ 2017-08-01

## 목표
- 4단계 퍼널 전환율 측정: Product View → Cart → Checkout → Purchase
- 가장 큰 이탈 지점(drop-off) 식별
- 디바이스별 퍼널 비교 (Desktop vs Mobile)
- 요일별 전환율 패턴
- 통계 검정: 디바이스별 전환율 차이의 유의성 (카이제곱 검정)

## eCommerceAction.action_type 매핑
| Code | Action |
|------|--------|
| '2' | Product Detail View (상품 조회) |
| '3' | Add to Cart (장바구니) |
| '5' | Checkout (결제 시작) |
| '6' | Purchase (구매 완료) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from google.cloud import bigquery

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

client = bigquery.Client()
print('BigQuery 연결 성공')

## 2.1 전체 4단계 퍼널

In [ ]:
query_funnel = """
WITH funnel AS (
  SELECT
    fullVisitorId, visitId,
    MAX(IF(hits.eCommerceAction.action_type = '2', 1, 0)) AS product_view,
    MAX(IF(hits.eCommerceAction.action_type = '3', 1, 0)) AS add_to_cart,
    MAX(IF(hits.eCommerceAction.action_type = '5', 1, 0)) AS checkout,
    MAX(IF(hits.eCommerceAction.action_type = '6', 1, 0)) AS purchase
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId, visitId
)
SELECT
  COUNT(*) AS total_sessions,
  COUNTIF(product_view = 1) AS product_view_sessions,
  COUNTIF(add_to_cart = 1) AS add_to_cart_sessions,
  COUNTIF(checkout = 1) AS checkout_sessions,
  COUNTIF(purchase = 1) AS purchase_sessions
FROM funnel
"""

df_funnel = client.query(query_funnel).to_dataframe()
df_funnel

In [ ]:
# 퍼널 시각화
stages = ['Product View', 'Add to Cart', 'Checkout', 'Purchase']
values = [
    df_funnel['product_view_sessions'].iloc[0],
    df_funnel['add_to_cart_sessions'].iloc[0],
    df_funnel['checkout_sessions'].iloc[0],
    df_funnel['purchase_sessions'].iloc[0]
]

# 전환율 계산
step_rates = []
for i in range(1, len(values)):
    rate = values[i] / values[i-1] * 100 if values[i-1] > 0 else 0
    step_rates.append(rate)

overall_rate = values[-1] / values[0] * 100 if values[0] > 0 else 0

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
bars = ax.barh(stages[::-1], [v for v in values[::-1]], color=colors[::-1])

for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + bar.get_width()*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Sessions')
ax.set_title('Ecommerce Funnel: Session Counts by Stage', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n--- Step Conversion Rates ---")
print(f"Product View → Add to Cart: {step_rates[0]:.2f}%")
print(f"Add to Cart → Checkout:     {step_rates[1]:.2f}%")
print(f"Checkout → Purchase:        {step_rates[2]:.2f}%")
print(f"\nOverall (View → Purchase):  {overall_rate:.2f}%")

In [ ]:
# 단계별 이탈율 시각화
drop_offs = []
for i in range(1, len(values)):
    drop = (values[i-1] - values[i]) / values[i-1] * 100
    drop_offs.append(drop)

fig, ax = plt.subplots(figsize=(8, 5))
drop_labels = ['View → Cart', 'Cart → Checkout', 'Checkout → Purchase']
bar_colors = ['#e15759' if d == max(drop_offs) else '#aab5c3' for d in drop_offs]
ax.bar(drop_labels, drop_offs, color=bar_colors)
ax.set_ylabel('Drop-off Rate (%)')
ax.set_title('Drop-off Rate by Funnel Stage')

for i, v in enumerate(drop_offs):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

biggest_drop_idx = drop_offs.index(max(drop_offs))
print(f"\n🔴 가장 큰 이탈 지점: {drop_labels[biggest_drop_idx]} ({drop_offs[biggest_drop_idx]:.1f}%)")

## 2.2 디바이스별 퍼널 비교

In [ ]:
query_device_funnel = """
WITH funnel AS (
  SELECT
    device.deviceCategory AS device,
    fullVisitorId, visitId,
    MAX(IF(hits.eCommerceAction.action_type = '2', 1, 0)) AS product_view,
    MAX(IF(hits.eCommerceAction.action_type = '3', 1, 0)) AS add_to_cart,
    MAX(IF(hits.eCommerceAction.action_type = '5', 1, 0)) AS checkout,
    MAX(IF(hits.eCommerceAction.action_type = '6', 1, 0)) AS purchase
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY device, fullVisitorId, visitId
)
SELECT
  device,
  COUNTIF(product_view = 1) AS pv,
  COUNTIF(add_to_cart = 1) AS atc,
  COUNTIF(checkout = 1) AS co,
  COUNTIF(purchase = 1) AS pur,
  ROUND(COUNTIF(add_to_cart = 1) * 100.0 / NULLIF(COUNTIF(product_view = 1), 0), 2) AS view_to_cart,
  ROUND(COUNTIF(purchase = 1) * 100.0 / NULLIF(COUNTIF(product_view = 1), 0), 2) AS overall_cvr
FROM funnel
GROUP BY device
ORDER BY pv DESC
"""

df_dev = client.query(query_device_funnel).to_dataframe()
df_dev

In [ ]:
# 디바이스별 전환율 비교 차트
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(df_dev['device']))
width = 0.35

ax.bar(x - width/2, df_dev['view_to_cart'], width, label='View → Cart %', color='#4e79a7')
ax.bar(x + width/2, df_dev['overall_cvr'], width, label='Overall CVR %', color='#e15759')

ax.set_xticks(x)
ax.set_xticklabels(df_dev['device'])
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Funnel Conversion by Device')
ax.legend()

for i, (v1, v2) in enumerate(zip(df_dev['view_to_cart'], df_dev['overall_cvr'])):
    ax.text(i - width/2, v1 + 0.3, f'{v1:.1f}%', ha='center', fontsize=9)
    ax.text(i + width/2, v2 + 0.1, f'{v2:.2f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 2.3 요일별 전환율

In [ ]:
query_dow = """
WITH funnel AS (
  SELECT
    PARSE_DATE('%Y%m%d', date) AS session_date,
    fullVisitorId, visitId,
    MAX(IF(hits.eCommerceAction.action_type = '2', 1, 0)) AS product_view,
    MAX(IF(hits.eCommerceAction.action_type = '6', 1, 0)) AS purchase
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY session_date, fullVisitorId, visitId
)
SELECT
  FORMAT_DATE('%A', session_date) AS day_of_week,
  EXTRACT(DAYOFWEEK FROM session_date) AS day_num,
  COUNTIF(product_view = 1) AS pv_sessions,
  COUNTIF(purchase = 1) AS purchase_sessions,
  ROUND(COUNTIF(purchase = 1) * 100.0 / NULLIF(COUNTIF(product_view = 1), 0), 2) AS cvr_pct
FROM funnel
GROUP BY day_of_week, day_num
ORDER BY day_num
"""

df_dow = client.query(query_dow).to_dataframe()
df_dow

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(df_dow['day_of_week'], df_dow['pv_sessions'], color='#4e79a7', alpha=0.6, label='Product View Sessions')
ax1.set_ylabel('Product View Sessions', color='#4e79a7')
ax1.tick_params(axis='y', labelcolor='#4e79a7')

ax2 = ax1.twinx()
ax2.plot(df_dow['day_of_week'], df_dow['cvr_pct'], color='#e15759',
         marker='o', linewidth=2, label='CVR %')
ax2.set_ylabel('Conversion Rate (%)', color='#e15759')
ax2.tick_params(axis='y', labelcolor='#e15759')

fig.suptitle('Conversion Rate by Day of Week', fontsize=14)
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.88))
plt.tight_layout()
plt.show()

## 2.4 구매까지 소요 시간

In [ ]:
query_time = """
WITH hit_times AS (
  SELECT
    fullVisitorId, visitId,
    MIN(IF(hits.eCommerceAction.action_type = '2', hits.time, NULL)) AS first_view_ms,
    MIN(IF(hits.eCommerceAction.action_type = '6', hits.time, NULL)) AS purchase_ms
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId, visitId
  HAVING first_view_ms IS NOT NULL AND purchase_ms IS NOT NULL
)
SELECT
  (purchase_ms - first_view_ms) / 1000 AS seconds_to_purchase
FROM hit_times
WHERE purchase_ms >= first_view_ms
"""

df_time = client.query(query_time).to_dataframe()

print(f"구매 세션 수: {len(df_time):,}")
print(f"평균 소요 시간: {df_time['seconds_to_purchase'].mean():.1f}초 ({df_time['seconds_to_purchase'].mean()/60:.1f}분)")
print(f"중앙값 소요 시간: {df_time['seconds_to_purchase'].median():.1f}초 ({df_time['seconds_to_purchase'].median()/60:.1f}분)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
# 1시간 이내만 표시 (아웃라이어 제외)
filtered = df_time[df_time['seconds_to_purchase'] <= 3600]['seconds_to_purchase'] / 60
ax.hist(filtered, bins=50, color='#4e79a7', edgecolor='white')
ax.axvline(filtered.median(), color='#e15759', linestyle='--', linewidth=2, label=f'Median: {filtered.median():.1f} min')
ax.set_xlabel('Minutes to Purchase')
ax.set_ylabel('Number of Sessions')
ax.set_title('Time from Product View to Purchase (within 1 hour)')
ax.legend()
plt.tight_layout()
plt.show()

## 2.5 통계 검정: Desktop vs Mobile 전환율 차이

In [ ]:
query_chi2 = """
WITH funnel AS (
  SELECT
    device.deviceCategory AS device,
    fullVisitorId, visitId,
    MAX(IF(hits.eCommerceAction.action_type = '2', 1, 0)) AS product_view,
    MAX(IF(hits.eCommerceAction.action_type = '6', 1, 0)) AS purchase
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY device, fullVisitorId, visitId
)
SELECT
  device,
  COUNTIF(product_view = 1 AND purchase = 1) AS purchased,
  COUNTIF(product_view = 1 AND purchase = 0) AS not_purchased,
  COUNTIF(product_view = 1) AS total_product_viewers
FROM funnel
WHERE device IN ('desktop', 'mobile')
GROUP BY device
ORDER BY device
"""

df_chi2 = client.query(query_chi2).to_dataframe()
df_chi2

In [ ]:
# 카이제곱 검정 (Chi-square test of independence)
contingency_table = df_chi2[['purchased', 'not_purchased']].values
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

# 효과 크기 (Cramer's V)
n = contingency_table.sum()
k = min(contingency_table.shape)
cramers_v = np.sqrt(chi2 / (n * (k - 1)))

print('='*60)
print('Chi-Square Test: Desktop vs Mobile Conversion Rate')
print('='*60)
print(f'H0: Desktop과 Mobile의 전환율에 차이가 없다')
print(f'H1: Desktop과 Mobile의 전환율에 유의미한 차이가 있다')
print(f'\nContingency Table:')
print(df_chi2.to_string(index=False))
print(f'\nChi-square statistic: {chi2:.4f}')
print(f'Degrees of freedom: {dof}')
print(f'p-value: {p_value:.2e}')
print(f"Cramer's V (effect size): {cramers_v:.4f}")
print(f'\nSignificance level: 0.05')

if p_value < 0.05:
    print(f'\n✅ Result: p < 0.05 → H0 기각. Desktop과 Mobile의 전환율 차이는 통계적으로 유의미하다.')
else:
    print(f'\n❌ Result: p >= 0.05 → H0 기각 불가. 유의미한 차이를 확인할 수 없다.')

# 전환율 계산
for _, row in df_chi2.iterrows():
    cvr = row['purchased'] / row['total_product_viewers'] * 100
    print(f"  {row['device']}: {cvr:.2f}% ({row['purchased']:,} / {row['total_product_viewers']:,})")

# ============================================================
# KEY FINDINGS (데이터에서 동적 생성)
# ============================================================
print("=" * 60)
print("KEY FINDINGS — Funnel Analysis")
print("=" * 60)

# Finding 1: 최대 이탈 지점
print(f"\n1. 최대 이탈 지점: {drop_labels[biggest_drop_idx]}")
print(f"   이탈율 {drop_offs[biggest_drop_idx]:.1f}%")
print(f"   → 이 단계의 UX 개선이 전환율에 가장 큰 임팩트")

# Finding 2: Desktop vs Mobile
desktop_cvr = df_chi2[df_chi2['device'] == 'desktop']
mobile_cvr = df_chi2[df_chi2['device'] == 'mobile']
if len(desktop_cvr) > 0 and len(mobile_cvr) > 0:
    d_rate = desktop_cvr.iloc[0]['purchased'] / desktop_cvr.iloc[0]['total_product_viewers'] * 100
    m_rate = mobile_cvr.iloc[0]['purchased'] / mobile_cvr.iloc[0]['total_product_viewers'] * 100
    ratio = d_rate / m_rate if m_rate > 0 else 0
    print(f"\n2. Desktop vs Mobile 전환율")
    print(f"   Desktop: {d_rate:.2f}%, Mobile: {m_rate:.2f}% (Desktop이 {ratio:.1f}x)")
    print(f"   Chi-square p-value: {p_value:.2e}, Cramer's V: {cramers_v:.4f}")
    if cramers_v < 0.1:
        print(f"   → 통계적으로 유의미하나 Cramer's V < 0.1이므로 실질적 효과 크기는 작음")
        print(f"   → 대규모 표본에서는 작은 차이도 유의미하게 나옴에 주의")
    else:
        print(f"   → 통계적으로 유의미하며 실질적 효과도 있음")

# Finding 3: 구매 소요 시간
median_min = df_time['seconds_to_purchase'].median() / 60
print(f"\n3. 구매 소요 시간 (상품 조회 → 구매)")
print(f"   중앙값: {median_min:.1f}분")
if median_min < 10:
    print(f"   → 충동 구매 패턴이 강함. 초기 제품 페이지 경험이 결정적")

# Finding 4: 순차 검증 차이
seq_cvr = seq_vals[-1] / seq_vals[0] * 100 if seq_vals[0] > 0 else 0
nseq_cvr = non_seq_vals[-1] / non_seq_vals[0] * 100 if non_seq_vals[0] > 0 else 0
print(f"\n4. 순차 검증의 영향")
print(f"   비순차 전환율: {nseq_cvr:.2f}% → 순차 전환율: {seq_cvr:.2f}%")
print(f"   → 순서를 무시하면 전환율을 {nseq_cvr - seq_cvr:.2f}%p 과대 추정")

In [ ]:
# 순차 vs 비순차 퍼널 비교 시각화
stages = ['Product View', 'Add to Cart', 'Checkout', 'Purchase']
non_seq = df_seq[df_seq['method'] == 'non_sequential'].iloc[0]
seq = df_seq[df_seq['method'] == 'sequential'].iloc[0]

non_seq_vals = [non_seq['step1_view'], non_seq['step2_cart'], non_seq['step3_checkout'], non_seq['step4_purchase']]
seq_vals = [seq['step1_view'], seq['step2_cart'], seq['step3_checkout'], seq['step4_purchase']]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(stages))
width = 0.35

bars1 = ax.bar(x - width/2, non_seq_vals, width, label='Non-Sequential (존재 여부만)', color='#aab5c3')
bars2 = ax.bar(x + width/2, seq_vals, width, label='Sequential (순서 검증)', color='#4e79a7')

ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel('Sessions')
ax.set_title('Sequential vs Non-Sequential Funnel Comparison', fontsize=14)
ax.legend()

for bar, val in zip(bars1, non_seq_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,.0f}', ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, seq_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# 차이 분석
print("\n--- Sequential vs Non-Sequential 차이 ---")
for i, stage in enumerate(stages):
    ns_val = non_seq_vals[i]
    s_val = seq_vals[i]
    diff_pct = (ns_val - s_val) / ns_val * 100 if ns_val > 0 else 0
    print(f"{stage}: {ns_val:,.0f} → {s_val:,.0f} (순차 검증 시 {diff_pct:.1f}% 감소)")

print(f"\n순차 퍼널 전체 전환율: {seq_vals[-1]/seq_vals[0]*100:.2f}%")
print(f"비순차 퍼널 전체 전환율: {non_seq_vals[-1]/non_seq_vals[0]*100:.2f}%")
print(f"→ 비순차 퍼널은 전환율을 과대 추정할 수 있음")

In [ ]:
query_sequential = """
WITH ordered_actions AS (
  SELECT
    fullVisitorId, visitId,
    MIN(IF(hits.eCommerceAction.action_type = '2', hits.hitNumber, NULL)) AS view_hit,
    MIN(IF(hits.eCommerceAction.action_type = '3', hits.hitNumber, NULL)) AS cart_hit,
    MIN(IF(hits.eCommerceAction.action_type = '5', hits.hitNumber, NULL)) AS checkout_hit,
    MIN(IF(hits.eCommerceAction.action_type = '6', hits.hitNumber, NULL)) AS purchase_hit
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
       UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId, visitId
)
SELECT
  'non_sequential' AS method,
  COUNTIF(view_hit IS NOT NULL) AS step1_view,
  COUNTIF(cart_hit IS NOT NULL) AS step2_cart,
  COUNTIF(checkout_hit IS NOT NULL) AS step3_checkout,
  COUNTIF(purchase_hit IS NOT NULL) AS step4_purchase
FROM ordered_actions
UNION ALL
SELECT
  'sequential' AS method,
  COUNTIF(view_hit IS NOT NULL) AS step1_view,
  COUNTIF(view_hit IS NOT NULL AND cart_hit IS NOT NULL
    AND cart_hit > view_hit) AS step2_cart,
  COUNTIF(view_hit IS NOT NULL AND cart_hit IS NOT NULL AND checkout_hit IS NOT NULL
    AND cart_hit > view_hit AND checkout_hit > cart_hit) AS step3_checkout,
  COUNTIF(view_hit IS NOT NULL AND cart_hit IS NOT NULL AND checkout_hit IS NOT NULL
    AND purchase_hit IS NOT NULL
    AND cart_hit > view_hit AND checkout_hit > cart_hit
    AND purchase_hit > checkout_hit) AS step4_purchase
FROM ordered_actions
"""

df_seq = client.query(query_sequential).to_dataframe()
df_seq

## Key Findings

1. **최대 이탈 지점**: Product View → Add to Cart 단계에서 가장 큰 이탈 발생
   - 상품 상세 페이지의 CTA 개선이 가장 큰 임팩트를 줄 수 있음

2. **Desktop vs Mobile**: Desktop의 전환율이 Mobile보다 유의미하게 높음
   - 카이제곱 검정 결과 p < 0.001 → 통계적으로 유의미
   - 모바일 UX 최적화가 매출 증대의 핵심 기회

3. **구매 소요 시간**: 대부분의 구매가 상품 조회 후 수분 이내에 발생
   - 충동 구매 패턴이 강함 → 초기 제품 페이지 경험이 결정적

4. **요일 효과**: 요일별 전환율에 차이가 존재
   - 프로모션 타이밍 최적화에 활용 가능

### Action Items
- 상품 상세 페이지에서 장바구니 추가 UX 개선 (가장 큰 이탈 지점)
- 모바일 체크아웃 프로세스 간소화
- 높은 전환율 요일에 프로모션 집중